# DQ 检查 — DWS 层（Rule 17）

Rule 17: 聚合守恒（DWS trip_count == fact 行数，允许 0.1% 误差）

FAIL 将抛出异常，阻断 ADS 下游任务。


## 1. SparkSession

In [ ]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterDWS")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT = f"{BASE}/data/warehouse/dwd/fact_trips"
DWS_OUT  = f"{BASE}/data/warehouse/dws/trips_daily"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


## 2. Rule 17：DWS 聚合守恒


In [ ]:
fact_count     = spark.read.parquet(DWD_FACT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo)
    & (F.col("pickup_day") == day)).count()
dws            = spark.read.parquet(DWS_OUT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo)
    & (F.col("pickup_day") == day))
dws_total      = dws.agg(F.sum("trip_count")).collect()[0][0]
diff_rate      = abs(dws_total - fact_count) / fact_count

passed = diff_rate < 0.001
mark   = "✅" if passed else "❌"
status = "PASS" if passed else "FAIL"
print(f"{mark} Rule 17 [DWS] DWS trips == fact rows (tolerance 0.1%)")
print(f"   fact_count={fact_count:,}  dws_total={dws_total:,}  diff={diff_rate:.4%}")

print(f"\n{'='*50}")
print(f"DQ Report (DWS)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")

# Persist Rule 17 result to dq_results (gate3_dws)
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
DQ_SCHEMA = StructType([
    StructField("check_date",   StringType(),  True),
    StructField("gate",         StringType(),  True),
    StructField("rule_id",      StringType(),  True),
    StructField("layer",        StringType(),  True),
    StructField("rule_type",    StringType(),  True),
    StructField("description",  StringType(),  True),
    StructField("status",       StringType(),  True),
    StructField("severity",     StringType(),  True),
    StructField("bad_count",    LongType(),    True),
    StructField("total_count",  LongType(),    True),
    StructField("threshold",    DoubleType(),  True),
    StructField("bad_rate",     DoubleType(),  True),
    StructField("pickup_year",  IntegerType(), True),
    StructField("pickup_month", IntegerType(), True),
    StructField("pickup_day",   IntegerType(), True),
])
DQ_RESULTS_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/warehouse/dq/dq_results")
row = [(
    TARGET_DATE, "gate3_dws", "17", "DWS", "conservation",
    "DWS trip_count == fact_trips (tolerance 0.1%)",
    status, "block",
    int(abs(dws_total - fact_count)), int(fact_count), 0.001,
    round(diff_rate, 6),
    yr, mo, day,
)]
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
(spark.createDataFrame(row, schema=DQ_SCHEMA)
     .write
     .partitionBy("gate", "pickup_year", "pickup_month", "pickup_day")
     .mode("overwrite")
     .parquet(DQ_RESULTS_PATH))
print(f"DQ Rule 17 persisted → {DQ_RESULTS_PATH}")

spark.stop()

if not passed:
    raise ValueError(
        f"DQ FAILED Rule 17: DWS aggregation mismatch {diff_rate:.4%}. "
        "Pipeline halted — fix DWS before proceeding to ADS."
    )
else:
    print("Rule 17 passed — proceeding to ADS.")

print("DQ after-DWS check complete.")